# 02 — Stage 1: Qwen3-8B QLoRA Fine-Tuning

**PsyDefDetect Pipeline — Stage 1**

This notebook fine-tunes `Qwen/Qwen3-8B` with QLoRA (4-bit NF4 quantisation + LoRA adapters) for
9-class Defence Mechanism Recognition Score (DMRS) classification on the PsyDef dataset.

| Section | Content | GPU needed? |
|---------|---------|-------------|
| 1 | Imports & Config | No |
| 2 | Prompt Construction | No |
| 3 | Data Loading & Tokenisation | No |
| 4 | Class Weights | No |
| 5 | Model Loading (QLoRA) | **Yes** |
| 6 | Training Loop (per fold) | **Yes** |
| 7 | Held-Out Routing Logic | No |
| 8 | Quick Evaluation (Macro-F1) | No |

> **Target GPU:** Vast.ai / RunPod with ≥24 GB VRAM (RTX 3090 or RTX 4090 recommended)

---
## Section 1 — Imports & Config

In [ ]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import json
import math
import os
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
from tqdm.auto import tqdm, trange

# ---------------------------------------------------------------------------
# Scikit-learn
# ---------------------------------------------------------------------------
from sklearn.metrics import (
    classification_report,
    f1_score,
)

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------------------
# HuggingFace Transformers
# ---------------------------------------------------------------------------
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

# ---------------------------------------------------------------------------
# PEFT / LoRA
# ---------------------------------------------------------------------------
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

# ---------------------------------------------------------------------------
# BitsAndBytes optimisers
# ---------------------------------------------------------------------------
import bitsandbytes as bnb

print(f"torch      : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ---------------------------------------------------------------------------
# Configuration dataclass
# ---------------------------------------------------------------------------
@dataclass
class CFG:
    # ── Reproducibility ────────────────────────────────────────────────────
    SEEDS: List[int] = field(default_factory=lambda: [42, 2026, 777])
    SEED: int = 42                      # primary seed used in this notebook

    # ── Models ─────────────────────────────────────────────────────────────
    MODEL_NAME_SMALL: str = "Qwen/Qwen3-4B"
    MODEL_NAME_MAIN:  str = "Qwen/Qwen3-8B"

    # ── Paths ──────────────────────────────────────────────────────────────
    DATA_DIR:     str = r"c:\Users\Public\Documents\TA\Dataset"
    ARTIFACT_DIR: str = r"c:\Users\Public\Documents\TA\Dataset\artifacts"

    # ── Cross-validation ───────────────────────────────────────────────────
    N_FOLDS: int = 5

    # ── Tokenisation ───────────────────────────────────────────────────────
    MAX_HISTORY_TURNS: int = 30
    MAX_LENGTH:        int = 1536

    # ── QLoRA hyper-parameters ─────────────────────────────────────────────
    LORA_R:              int   = 128
    LORA_ALPHA:          int   = 256
    LORA_DROPOUT:        float = 0.1
    LORA_TARGET_MODULES: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "score"
    ])

    # ── Optimiser ──────────────────────────────────────────────────────────
    LR:           float = 1.2e-4
    WEIGHT_DECAY: float = 0.01
    GRAD_CLIP:    float = 0.3

    # ── Scheduler ──────────────────────────────────────────────────────────
    WARMUP_RATIO: float = 0.08

    # ── Loss ───────────────────────────────────────────────────────────────
    LABEL_SMOOTHING: float = 0.05

    # ── Training ───────────────────────────────────────────────────────────
    MAX_EPOCHS:          int = 10
    EFFECTIVE_BATCH_SIZE: int = 16
    PHYSICAL_BATCH_SIZE:  int = 2      # adjust based on actual VRAM

    # ── Labels ─────────────────────────────────────────────────────────────
    NUM_LABELS: int = 9
    LABEL_NAMES: List[str] = field(default_factory=lambda: [
        "No Defense",             # 0
        "Action",                 # 1
        "Major Image-Distorting", # 2
        "Disavowal",              # 3
        "Minor Image-Distorting", # 4
        "Neurotic",               # 5
        "Obsessional",            # 6
        "High-Adaptive",          # 7
        "Needs More Info",        # 8
    ])


cfg = CFG()

# Derived artifact directories
ARTIFACT_DIR   = Path(cfg.ARTIFACT_DIR)
SPLITS_DIR     = ARTIFACT_DIR / "splits"
STAGE1_DIR     = ARTIFACT_DIR / "stage1_qwen3_8b" / f"seed_{cfg.SEED}"
RESULTS_DIR    = ARTIFACT_DIR / "results"

for d in [SPLITS_DIR, STAGE1_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Artifact directories created/verified.")
print(f"  splits  : {SPLITS_DIR}")
print(f"  stage1  : {STAGE1_DIR}")
print(f"  results : {RESULTS_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Seed utility
# ---------------------------------------------------------------------------
def set_seed(seed: int = 42) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(cfg.SEED)
print(f"Seeds set to {cfg.SEED}.")

---
## Section 2 — Prompt Construction

We build a structured prompt with three parts:
1. **Role instruction** + full DMRS level definitions (labels 0–8).
2. **Dialogue history** — the last `max_history_turns` turns from `dialogue[:-1]`
   (excluding the current turn so no data leakage). Speaker prefixes: `SEEKER:` / `SUPPORTER:`.
3. **Current utterance** — `(Turn N)\nSEEKER: {current_text}`.

In [ ]:
# ---------------------------------------------------------------------------
# DMRS level definitions (inline, authoritative)
# ---------------------------------------------------------------------------
DMRS_DEFINITIONS = """\
Defence Mechanism Rating Scale (DMRS) — Level Definitions
==========================================================
Level 0 — No Defense:
  The speaker does not employ any recognisable psychological defence mechanism.
  The response is direct, transparent, and emotionally congruent with the
  situation. No distortion, avoidance, or coping strategy is detectable.

Level 1 — Action (Lowest Adaptive Level):
  Defences expressed through behaviour rather than cognition or affect.
  Includes acting out (impulse-driven behaviour to avoid distressing feelings),
  passive aggression (indirect expression of hostility), and help-rejecting
  complaining (seeking help while persistently refusing it).

Level 2 — Major Image-Distorting (Borderline Level):
  Severe distortion of self-image or others. Includes splitting (all-good or
  all-bad), projective identification (attributing unacceptable feelings to
  others while remaining identified with them), and autistic fantasy (retreat
  into fantasy to avoid real relationships or tasks).

Level 3 — Disavowal:
  Keeping distressing thoughts, feelings, or impulses out of awareness without
  gross reality distortion. Includes denial (refusal to accept reality),
  projection (attributing own unacceptable feelings to others), and
  rationalisation (constructing logical explanations to justify unacceptable
  motives).

Level 4 — Minor Image-Distorting (Narcissistic Level):
  Moderate distortion of self or others to regulate self-esteem. Includes
  omnipotence (inflated sense of personal power), devaluation (diminishing
  others to feel superior), and idealisation (exaggerated positive view of
  others to gain sense of safety or worth).

Level 5 — Neurotic Defences:
  More adaptive than disavowal but still involve significant intrapsychic
  distortion. Includes repression (involuntary forgetting of distressing
  material), reaction formation (expressing the opposite of a true feeling),
  and displacement (redirecting feelings from their true target to a safer one).

Level 6 — Obsessional Defences:
  Defences that use thinking, ritual, or isolation to manage affect. Includes
  intellectualisation (using abstract thought to distance from emotions),
  isolation of affect (separating feeling from related ideas), and
  undoing (symbolic acts that counteract previous thoughts or behaviours).

Level 7 — High-Adaptive (Mature) Defences:
  Flexible, context-appropriate coping strategies that maximise adaptation
  while allowing conscious awareness of impulses and feelings. Includes
  affiliation (seeking support from others), altruism (channelling drives into
  prosocial actions), anticipation (realistic planning for future distress),
  humour (using comedy to cope with difficulty), self-assertion (expressing
  feelings and thoughts directly), self-observation (reflection on own
  reactions), sublimation (redirecting impulses into constructive activity),
  and suppression (conscious decision to defer distressing thoughts).

Level 8 — Needs More Info:
  The available utterance is too short, ambiguous, or lacking context to
  reliably assign a DMRS level. Additional turns of dialogue are required
  before a defensible rating can be made.
"""

ROLE_INSTRUCTION = (
    "You are an expert clinical psychologist specialising in defence mechanism analysis "
    "using the Defence Mechanism Rating Scale (DMRS). "
    "Your task is to read a counselling dialogue and classify the SEEKER's CURRENT "
    "utterance (shown at the end) into exactly one of the nine DMRS levels (0–8) "
    "defined below.\n\n"
)


def build_prompt(sample: dict, max_history_turns: int = 30) -> str:
    """
    Build a structured classification prompt for a PsyDef sample.

    Parameters
    ----------
    sample : dict
        One entry from train.json / test.json with keys
        ``id``, ``dialogue_id``, ``dialogue``, ``current_text``, ``label``.
    max_history_turns : int
        Maximum number of prior turns (from dialogue[:-1]) to include.

    Returns
    -------
    str
        The full prompt string ready for tokenisation.
    """
    dialogue     = sample["dialogue"]
    current_text = sample["current_text"]

    # Safety: use only dialogue[:-1] — no future leakage
    history = dialogue[:-1]

    # Sanity assertion
    for turn in history:
        assert turn["text"] != current_text or turn["speaker"] != "seeker", (
            "Future turn detected in history! This is a data leakage bug."
        )

    # ── Part 1: Role + definitions ────────────────────────────────────────
    part1 = ROLE_INSTRUCTION + DMRS_DEFINITIONS

    # ── Part 2: Dialogue history ──────────────────────────────────────────
    recent_history = history[-max_history_turns:] if history else []

    if not recent_history:
        history_str = "[Start of conversation]\n"
    else:
        lines = []
        for turn in recent_history:
            speaker_tag = "SEEKER" if turn["speaker"] == "seeker" else "SUPPORTER"
            lines.append(f"{speaker_tag}: {turn['text']}")
        history_str = "\n".join(lines) + "\n"

    part2 = "--- Dialogue History ---\n" + history_str

    # ── Part 3: Current utterance ─────────────────────────────────────────
    turn_number = len(dialogue)
    part3 = (
        "--- Current Utterance to Classify ---\n"
        f"(Turn {turn_number})\n"
        f"SEEKER: {current_text}\n\n"
        "Your classification (integer 0–8):"
    )

    return part1 + "\n" + part2 + "\n" + part3


print("build_prompt() defined. Quick smoke-test on a dummy sample:")
dummy = {
    "id": "dummy",
    "dialogue_id": "d0",
    "dialogue": [
        {"speaker": "supporter", "text": "How can I help?"},
        {"speaker": "seeker",    "text": "I feel overwhelmed."},
    ],
    "current_text": "I feel overwhelmed.",
    "label": 7,
}
p = build_prompt(dummy)
print(f"  Prompt length (chars): {len(p)}")
print(f"  First 200 chars:\n{p[:200]}")

---
## Section 3 — Data Loading & Tokenisation

1. Load `train.json` and `test.json`.
2. Merge fold assignments from `artifacts/splits/train_folds.parquet`.
3. Tokenise all prompts and report coverage stats at `max_length=1536`.
4. Define `PsyDefDataset` with dynamic padding (pad to longest-in-batch, multiple of 8).

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Load raw data
# ---------------------------------------------------------------------------
DATA_DIR = Path(cfg.DATA_DIR)

with open(DATA_DIR / "train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DATA_DIR / "test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Train samples : {len(train_data)}")
print(f"Test  samples : {len(test_data)}")
print(f"Train keys    : {list(train_data[0].keys())}")

In [ ]:
# ---------------------------------------------------------------------------
# 3.2  Load fold assignments
# ---------------------------------------------------------------------------
folds_path = SPLITS_DIR / "train_folds.parquet"
fold_df = pd.read_parquet(folds_path)
print(f"Fold DF shape : {fold_df.shape}")
print(fold_df.head())
print(f"\nFold counts:\n{fold_df['fold_id'].value_counts().sort_index()}")

In [ ]:
# ---------------------------------------------------------------------------
# 3.3  Merge fold_id into train_data
# ---------------------------------------------------------------------------
id_to_fold = dict(zip(fold_df["id"], fold_df["fold_id"]))

for s in train_data:
    s["fold_id"] = id_to_fold.get(s["id"], -1)

missing_folds = sum(1 for s in train_data if s["fold_id"] == -1)
print(f"Samples missing fold assignment: {missing_folds}")

In [ ]:
# ---------------------------------------------------------------------------
# 3.4  Build all prompts
# ---------------------------------------------------------------------------
print("Building prompts for train set ...")
for s in tqdm(train_data, desc="train prompts"):
    s["prompt"] = build_prompt(s, max_history_turns=cfg.MAX_HISTORY_TURNS)

print("Building prompts for test set ...")
for s in tqdm(test_data, desc="test prompts"):
    s["prompt"] = build_prompt(s, max_history_turns=cfg.MAX_HISTORY_TURNS)

print("Done.")

In [ ]:
# ---------------------------------------------------------------------------
# 3.5  Tokenise & report coverage stats
# ---------------------------------------------------------------------------
print(f"Loading tokeniser: {cfg.MODEL_NAME_MAIN} ...")
tokenizer = AutoTokenizer.from_pretrained(
    cfg.MODEL_NAME_MAIN,
    trust_remote_code=True,
    padding_side="left",   # for decoder-style models; adjust if needed
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokeniser loaded. Computing token lengths for all prompts ...")

all_prompts = [s["prompt"] for s in train_data] + [s["prompt"] for s in test_data]

token_lengths = []
BATCH = 128
for i in trange(0, len(all_prompts), BATCH, desc="tokenising"):
    batch = all_prompts[i : i + BATCH]
    enc = tokenizer(
        batch,
        truncation=False,
        padding=False,
        return_attention_mask=False,
    )
    token_lengths.extend(len(ids) for ids in enc["input_ids"])

token_lengths = np.array(token_lengths)
train_len = token_lengths[: len(train_data)]
test_len  = token_lengths[len(train_data) :]

MAX_LEN = cfg.MAX_LENGTH

print("\n=== Token-length statistics ===")
for name, arr in [("TRAIN", train_len), ("TEST", test_len), ("ALL", token_lengths)]:
    pct_within = (arr <= MAX_LEN).mean() * 100
    print(
        f"  {name:5s}  min={arr.min():5d}  median={int(np.median(arr)):5d}  "
        f"p95={int(np.percentile(arr, 95)):5d}  max={arr.max():5d}  "
        f"within_{MAX_LEN}={pct_within:.1f}%"
    )

In [ ]:
# ---------------------------------------------------------------------------
# 3.6  PsyDefDataset — dynamic padding, multiple-of-8
# ---------------------------------------------------------------------------
def _round_up_to_multiple(n: int, multiple: int = 8) -> int:
    return multiple * math.ceil(n / multiple)


class PsyDefDataset(Dataset):
    """
    PyTorch Dataset for PsyDef counselling dialogue samples.

    Returns tokenised ``{input_ids, attention_mask, labels}`` dicts.
    Dynamic padding (pad to longest-in-batch, multiple of 8) is handled
    by the accompanying ``collate_fn``.
    """

    def __init__(
        self,
        samples: List[dict],
        tokenizer,
        max_length: int = 1536,
        has_labels: bool = True,
    ):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        s   = self.samples[idx]
        enc = self.tokenizer(
            s["prompt"],
            max_length=self.max_length,
            truncation=True,
            padding=False,
            return_attention_mask=True,
        )
        item = {
            "input_ids":      torch.tensor(enc["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if self.has_labels:
            item["labels"] = torch.tensor(s["label"], dtype=torch.long)
        return item


def collate_fn(batch: List[dict], pad_token_id: int = 0) -> dict:
    """
    Dynamic-padding collate function.
    Pads sequences to the longest in the batch, rounded up to a multiple of 8.
    Uses left-padding so the last token is always the EOS / classification token.
    """
    max_len = max(item["input_ids"].size(0) for item in batch)
    max_len = _round_up_to_multiple(max_len, 8)

    input_ids_list      = []
    attention_mask_list = []
    labels_list         = []
    has_labels          = "labels" in batch[0]

    for item in batch:
        seq_len  = item["input_ids"].size(0)
        pad_len  = max_len - seq_len

        # Left-pad
        input_ids_list.append(
            torch.cat([torch.full((pad_len,), pad_token_id, dtype=torch.long),
                       item["input_ids"]])
        )
        attention_mask_list.append(
            torch.cat([torch.zeros(pad_len, dtype=torch.long),
                       item["attention_mask"]])
        )
        if has_labels:
            labels_list.append(item["labels"])

    out = {
        "input_ids":      torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_mask_list),
    }
    if has_labels:
        out["labels"] = torch.stack(labels_list)
    return out


print("PsyDefDataset and collate_fn defined.")

---
## Section 4 — Class Weights

Inverse-square-root weighting normalised to sum to 1:

$$w_c = \frac{1/\sqrt{n_c}}{\sum_{i=0}^{8} 1/\sqrt{n_i}}$$

This down-weights frequent classes (especially class 7 — High-Adaptive with 968 samples)
and up-weights rare classes (class 8 — Needs More Info with only 28 samples).

In [ ]:
def compute_class_weights(labels_in_fold: List[int]) -> torch.Tensor:
    """
    Compute inverse-square-root class weights.

    Parameters
    ----------
    labels_in_fold : list of int
        Class labels for the training partition (not validation).

    Returns
    -------
    torch.Tensor of shape (9,)
        Normalised class weights.
    """
    n_classes = cfg.NUM_LABELS
    counts = np.zeros(n_classes, dtype=np.float64)
    for lbl in labels_in_fold:
        counts[lbl] += 1

    # Replace zero counts with 1 to avoid division by zero
    counts = np.where(counts == 0, 1, counts)

    inv_sqrt = 1.0 / np.sqrt(counts)
    weights  = inv_sqrt / inv_sqrt.sum()

    return torch.tensor(weights, dtype=torch.float32)


# Quick demonstration with full training set
all_train_labels = [s["label"] for s in train_data]
demo_weights = compute_class_weights(all_train_labels)

print("Class weights (full training set):")
for i, (name, w) in enumerate(zip(cfg.LABEL_NAMES, demo_weights.tolist())):
    n = all_train_labels.count(i)
    print(f"  [{i}] {name:25s}  n={n:5d}  weight={w:.4f}")

---
## Section 5 — Model Loading (QLoRA)

We load `Qwen/Qwen3-8B` in 4-bit NF4 quantisation via `bitsandbytes`, then wrap it
with LoRA adapters (rank 128, alpha 256) targeting all projection layers.

> **# === GPU REQUIRED ===**  Cells in this section and Section 6 require a CUDA GPU with ≥24 GB VRAM.

In [ ]:
# === GPU REQUIRED ===

def load_qlora_model(
    model_name: str = None,
    num_labels: int = 9,
):
    """
    Load a causal LM as a sequence classifier with QLoRA.

    Steps
    -----
    1. Build BitsAndBytesConfig (4-bit NF4, double-quant, bfloat16 compute).
    2. Load AutoModelForSequenceClassification with the BnB config.
    3. Apply ``prepare_model_for_kbit_training()``.
    4. Apply LoRA via ``LoraConfig`` and ``get_peft_model()``.
    5. Enable gradient checkpointing.

    Returns
    -------
    peft_model : PeftModel
    """
    if model_name is None:
        model_name = cfg.MODEL_NAME_MAIN

    # ── BitsAndBytes 4-bit config ─────────────────────────────────────────
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # ── Base model ────────────────────────────────────────────────────────
    print(f"Loading {model_name} in 4-bit NF4 ...")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )

    # Set pad_token_id on the config to avoid generation warnings
    if base_model.config.pad_token_id is None:
        base_model.config.pad_token_id = tokenizer.pad_token_id

    # ── kbit training prep ────────────────────────────────────────────────
    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=True,
    )

    # ── LoRA config ───────────────────────────────────────────────────────
    lora_config = LoraConfig(
        r=cfg.LORA_R,
        lora_alpha=cfg.LORA_ALPHA,
        lora_dropout=cfg.LORA_DROPOUT,
        target_modules=cfg.LORA_TARGET_MODULES,
        task_type=TaskType.SEQ_CLS,
        bias="none",
    )

    peft_model = get_peft_model(base_model, lora_config)

    # Gradient checkpointing (saves VRAM at the cost of some compute)
    peft_model.enable_input_require_grads()  # required with gradient checkpointing
    peft_model.gradient_checkpointing_enable()

    peft_model.print_trainable_parameters()
    return peft_model


print("load_qlora_model() defined. Call it inside the training loop per fold.")

---
## Section 6 — Training Loop (per fold)

**Key design decisions:**
- `WeightedLabelSmoothingLoss` — class-weighted cross-entropy with ε=0.05 label smoothing.
- `bnb.optim.PagedAdamW8bit` — 8-bit paged AdamW to minimise VRAM footprint.
- Cosine schedule with linear warmup (`warmup_ratio=0.08`).
- Gradient clipping at 0.3 for stability.
- Gradient accumulation to reach effective batch size 16.
- **Resumable**: each fold checks for a pre-existing best checkpoint and skips training if found.
- OOF logits + hidden states + test logits are written after **every fold** to guard against crashes.

> **# === GPU REQUIRED ===**

In [ ]:
# === GPU REQUIRED ===

# ---------------------------------------------------------------------------
# Weighted label-smoothing loss
# ---------------------------------------------------------------------------
class WeightedLabelSmoothingLoss(nn.Module):
    """
    Cross-entropy loss with per-class weights and label smoothing.

    For a sample with true class c and smoothing ε:
      - true-class target  : 1 - ε
      - other-class targets: ε / (K - 1)   (K = num_classes)
    The final per-sample loss is then multiplied by w_c (the weight of class c).

    Parameters
    ----------
    weight : torch.Tensor of shape (K,)
        Class weights.
    num_classes : int
    epsilon : float
        Label-smoothing factor (0 = standard cross-entropy).
    """

    def __init__(
        self,
        weight: torch.Tensor,
        num_classes: int = 9,
        epsilon: float = 0.05,
    ):
        super().__init__()
        self.register_buffer("weight", weight)
        self.num_classes = num_classes
        self.epsilon     = epsilon

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        logits  : (B, K) — raw model outputs (before softmax)
        targets : (B,)   — integer class indices

        Returns
        -------
        scalar loss
        """
        K   = self.num_classes
        eps = self.epsilon

        log_probs = torch.log_softmax(logits.float(), dim=-1)  # (B, K)

        # Smooth targets
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs, eps / (K - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - eps)

        # Per-sample NLL
        per_sample_loss = -(smooth_targets * log_probs).sum(dim=-1)  # (B,)

        # Weight by true-class weight
        sample_weights = self.weight[targets]  # (B,)
        loss = (per_sample_loss * sample_weights).mean()
        return loss


print("WeightedLabelSmoothingLoss defined.")

In [ ]:
# === GPU REQUIRED ===

# ---------------------------------------------------------------------------
# Inference helper (returns logits + last hidden states)
# ---------------------------------------------------------------------------
@torch.no_grad()
def run_inference(
    model,
    loader: DataLoader,
    device: torch.device,
    desc: str = "inference",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run forward passes over a DataLoader, collecting logits and hidden states.

    Returns
    -------
    logits  : np.ndarray of shape (N, 9)
    hidden  : np.ndarray of shape (N, hidden_dim)  — last-token hidden states
    """
    model.eval()
    all_logits  = []
    all_hidden  = []

    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        logits = outputs.logits.float().cpu().numpy()       # (B, 9)

        # Last-layer hidden state at the classification position
        # For causal LM, the last non-padding token is the classification token.
        last_hidden = outputs.hidden_states[-1]              # (B, L, D)
        # Gather last non-padding position per sample
        seq_lens = attention_mask.sum(dim=1) - 1             # (B,) indices
        batch_size = last_hidden.size(0)
        hidden_vecs = last_hidden[
            torch.arange(batch_size, device=device), seq_lens
        ].float().cpu().numpy()                              # (B, D)

        all_logits.append(logits)
        all_hidden.append(hidden_vecs)

    return np.vstack(all_logits), np.vstack(all_hidden)


print("run_inference() defined.")

In [ ]:
# === GPU REQUIRED ===

# ---------------------------------------------------------------------------
# Main training loop — 5-fold cross-validation
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {DEVICE}")

ACCUM_STEPS = max(1, cfg.EFFECTIVE_BATCH_SIZE // cfg.PHYSICAL_BATCH_SIZE)
print(f"Physical batch : {cfg.PHYSICAL_BATCH_SIZE}")
print(f"Accum steps    : {ACCUM_STEPS}")
print(f"Effective batch: {cfg.PHYSICAL_BATCH_SIZE * ACCUM_STEPS}")

# Pre-allocate output arrays
N_TRAIN   = len(train_data)
N_TEST    = len(test_data)
N_FOLDS   = cfg.N_FOLDS
N_CLASSES = cfg.NUM_LABELS

oof_logits           = np.full((N_TRAIN, N_CLASSES), np.nan, dtype=np.float32)
oof_hidden           = None   # shape determined by first fold
test_logits_by_fold  = np.full((N_FOLDS, N_TEST, N_CLASSES), np.nan, dtype=np.float32)
test_hidden_by_fold  = None   # shape determined by first fold

# Map sample id -> position in train_data
id_to_train_idx = {s["id"]: i for i, s in enumerate(train_data)}

set_seed(cfg.SEED)

# ─────────────────────────────────────────────────────────────────────────
for fold_k in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f" FOLD {fold_k}")
    print(f"{'='*60}")

    fold_best_dir = STAGE1_DIR / f"fold_{fold_k}_best"
    oof_npy       = STAGE1_DIR / "oof_logits.npy"
    oof_hid_npy   = STAGE1_DIR / "oof_hidden.npy"
    tst_npy       = STAGE1_DIR / "test_logits_by_fold.npy"

    # ── Split ────────────────────────────────────────────────────────────
    train_samples_k = [s for s in train_data if s["fold_id"] != fold_k]
    val_samples_k   = [s for s in train_data if s["fold_id"] == fold_k]
    print(f"  Train: {len(train_samples_k)}  Val: {len(val_samples_k)}")

    # ── Resumable: skip if best checkpoint already exists ─────────────────
    if fold_best_dir.exists():
        print(f"  [SKIP] Checkpoint found at {fold_best_dir}. Loading saved logits.")
        # Restore previously-saved arrays
        if oof_npy.exists():
            oof_logits = np.load(oof_npy)
        if oof_hid_npy.exists():
            oof_hidden = np.load(oof_hid_npy)
        if tst_npy.exists():
            test_logits_by_fold = np.load(tst_npy)
        continue

    # ── Class weights ─────────────────────────────────────────────────────
    train_labels_k = [s["label"] for s in train_samples_k]
    class_weights  = compute_class_weights(train_labels_k).to(DEVICE)

    # ── DataLoaders ───────────────────────────────────────────────────────
    train_ds = PsyDefDataset(train_samples_k, tokenizer, cfg.MAX_LENGTH)
    val_ds   = PsyDefDataset(val_samples_k,   tokenizer, cfg.MAX_LENGTH)
    test_ds  = PsyDefDataset(test_data,        tokenizer, cfg.MAX_LENGTH, has_labels=False)

    _collate = lambda batch: collate_fn(batch, pad_token_id=tokenizer.pad_token_id)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.PHYSICAL_BATCH_SIZE,
        shuffle=True,
        collate_fn=_collate,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.PHYSICAL_BATCH_SIZE * 2,
        shuffle=False,
        collate_fn=_collate,
        num_workers=2,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.PHYSICAL_BATCH_SIZE * 2,
        shuffle=False,
        collate_fn=_collate,
        num_workers=2,
        pin_memory=True,
    )

    # ── Model ─────────────────────────────────────────────────────────────
    set_seed(cfg.SEED)
    model = load_qlora_model(cfg.MODEL_NAME_MAIN, num_labels=N_CLASSES)

    # ── Loss ──────────────────────────────────────────────────────────────
    criterion = WeightedLabelSmoothingLoss(
        weight=class_weights,
        num_classes=N_CLASSES,
        epsilon=cfg.LABEL_SMOOTHING,
    ).to(DEVICE)

    # ── Optimiser (8-bit paged AdamW) ─────────────────────────────────────
    optimizer = bnb.optim.PagedAdamW8bit(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY,
    )

    # ── Scheduler ─────────────────────────────────────────────────────────
    total_steps  = (len(train_loader) // ACCUM_STEPS) * cfg.MAX_EPOCHS
    warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    print(f"  Total optimiser steps : {total_steps}")
    print(f"  Warmup steps          : {warmup_steps}")

    # ── Training epochs ───────────────────────────────────────────────────
    best_macro_f1   = -1.0
    global_step     = 0

    for epoch in range(cfg.MAX_EPOCHS):
        model.train()
        epoch_loss   = 0.0
        n_batches    = 0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{cfg.MAX_EPOCHS}", leave=False)
        for step, batch in enumerate(pbar):
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            loss = criterion(outputs.logits, labels) / ACCUM_STEPS
            loss.backward()

            epoch_loss += loss.item() * ACCUM_STEPS
            n_batches  += 1

            if (step + 1) % ACCUM_STEPS == 0:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            pbar.set_postfix({"loss": f"{epoch_loss/n_batches:.4f}"})

        avg_loss = epoch_loss / n_batches

        # ── Validation ────────────────────────────────────────────────────
        val_logits, val_hidden = run_inference(model, val_loader, DEVICE, desc="  val")
        val_preds = val_logits.argmax(axis=1)
        val_true  = np.array([s["label"] for s in val_samples_k])

        # Macro-F1 on classes 1–8 (exclude class 0 for primary metric)
        macro_f1_1_8 = f1_score(
            val_true, val_preds,
            labels=list(range(1, 9)),
            average="macro",
            zero_division=0,
        )
        macro_f1_0_8 = f1_score(
            val_true, val_preds,
            average="macro",
            zero_division=0,
        )

        print(
            f"  Epoch {epoch+1:2d} | loss={avg_loss:.4f} | "
            f"val F1(1-8)={macro_f1_1_8:.4f} | val F1(0-8)={macro_f1_0_8:.4f}"
        )

        # ── Save best checkpoint ──────────────────────────────────────────
        if macro_f1_1_8 > best_macro_f1:
            best_macro_f1  = macro_f1_1_8
            best_val_logits = val_logits.copy()
            best_val_hidden = val_hidden.copy()
            fold_best_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(fold_best_dir))
            tokenizer.save_pretrained(str(fold_best_dir))
            print(f"  ✓ New best F1(1-8): {best_macro_f1:.4f} — checkpoint saved.")

    # ── Post-fold: test inference ─────────────────────────────────────────
    print(f"  Running test inference with best checkpoint ...")
    # Reload best adapter
    del model
    torch.cuda.empty_cache()

    best_base = load_qlora_model(cfg.MODEL_NAME_MAIN, num_labels=N_CLASSES)
    # Merge saved LoRA weights
    best_model = PeftModel.from_pretrained(best_base.base_model.model, str(fold_best_dir))
    best_model.to(DEVICE)

    test_logits_k, test_hidden_k = run_inference(
        best_model, test_loader, DEVICE, desc=f"  test fold {fold_k}"
    )

    # ── Write OOF logits / hidden ─────────────────────────────────────────
    val_indices = [id_to_train_idx[s["id"]] for s in val_samples_k]
    oof_logits[val_indices] = best_val_logits

    hidden_dim = best_val_hidden.shape[1]
    if oof_hidden is None:
        oof_hidden          = np.full((N_TRAIN, hidden_dim), np.nan, dtype=np.float32)
        test_hidden_by_fold = np.full((N_FOLDS, N_TEST, hidden_dim), np.nan, dtype=np.float32)

    oof_hidden[val_indices] = best_val_hidden

    # ── Write test logits ─────────────────────────────────────────────────
    test_logits_by_fold[fold_k] = test_logits_k
    test_hidden_by_fold[fold_k] = test_hidden_k

    # ── Persist to disk after every fold ─────────────────────────────────
    np.save(oof_npy,  oof_logits)
    np.save(oof_hid_npy, oof_hidden)
    np.save(tst_npy, test_logits_by_fold)
    print(f"  Fold {fold_k} artefacts saved to {STAGE1_DIR}")

    del best_model
    torch.cuda.empty_cache()

print("\n=== All folds complete ===")

---
## Section 7 — Held-Out Routing Logic

For test samples whose `dialogue_id` appeared in training, we select the logit vector
from the **fold where that dialogue was held out** — avoiding any leakage from the
fold that saw those dialogues during training.

For test samples whose `dialogue_id` never appeared in training, we use the
**ensemble mean** of all 5 fold logits (since no fold has an advantage).

We also save a naive 5-fold average as the "inflated" comparison baseline.

In [ ]:
# ---------------------------------------------------------------------------
# Reload arrays from disk (safe to run after a restart)
# ---------------------------------------------------------------------------
oof_npy     = STAGE1_DIR / "oof_logits.npy"
tst_npy     = STAGE1_DIR / "test_logits_by_fold.npy"

oof_logits           = np.load(oof_npy)
test_logits_by_fold  = np.load(tst_npy)

print(f"oof_logits shape          : {oof_logits.shape}")
print(f"test_logits_by_fold shape : {test_logits_by_fold.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Held-out fold look-up helper
# ---------------------------------------------------------------------------
def get_held_out_fold(dialogue_id: str, fold_df: pd.DataFrame) -> int:
    """
    Return the fold_id in which ``dialogue_id`` was the held-out validation set.

    With StratifiedGroupKFold, every unique dialogue_id belongs to exactly one
    fold (all utterances of a dialogue land in the same fold).  The *held-out*
    fold for a dialogue_id is therefore the fold_id assigned to its rows.

    Returns -1 if the dialogue_id is not found in fold_df (test-only dialogues).
    """
    rows = fold_df[fold_df["dialogue_id"] == dialogue_id]
    if rows.empty:
        return -1
    return int(rows["fold_id"].iloc[0])


# ---------------------------------------------------------------------------
# Assemble routed test logits
# ---------------------------------------------------------------------------
print("Assembling test_logits_routed.npy ...")

test_logits_routed = np.full((N_TEST, N_CLASSES), np.nan, dtype=np.float32)
n_routed   = 0
n_ensemble = 0

for i, s in enumerate(tqdm(test_data, desc="routing")):
    held_fold = get_held_out_fold(s["dialogue_id"], fold_df)

    if held_fold == -1:
        # dialogue_id only in test — use ensemble mean
        test_logits_routed[i] = test_logits_by_fold[:, i, :].mean(axis=0)
        n_ensemble += 1
    else:
        # Use the fold that held this dialogue out
        test_logits_routed[i] = test_logits_by_fold[held_fold, i, :]
        n_routed += 1

assert not np.isnan(test_logits_routed).any(), "Some test logits are still NaN!"

routed_path  = STAGE1_DIR / "test_logits_routed.npy"
np.save(routed_path, test_logits_routed)

print(f"  Routed  (held-out fold)  : {n_routed} samples")
print(f"  Ensemble (test-only dlg) : {n_ensemble} samples")
print(f"  Saved → {routed_path}")

In [ ]:
# ---------------------------------------------------------------------------
# 5-fold average baseline ("inflated" — dialogues seen in training)
# ---------------------------------------------------------------------------
test_logits_5fold_avg = test_logits_by_fold.mean(axis=0)   # (472, 9)

avg_path = STAGE1_DIR / "test_logits_5fold_avg.npy"
np.save(avg_path, test_logits_5fold_avg)

print(f"test_logits_5fold_avg shape : {test_logits_5fold_avg.shape}")
print(f"Saved → {avg_path}")

---
## Section 8 — Quick Evaluation (Macro-F1 Reporting)

Using the OOF logits (each validation sample was predicted by the fold that never
saw its dialogue during training), we report unbiased performance estimates.

In [ ]:
# ---------------------------------------------------------------------------
# 8.1  OOF macro-F1
# ---------------------------------------------------------------------------
oof_preds = oof_logits.argmax(axis=1)
oof_true  = np.array([s["label"] for s in train_data])

macro_f1_1_8 = f1_score(
    oof_true, oof_preds,
    labels=list(range(1, 9)),
    average="macro",
    zero_division=0,
)
macro_f1_0_8 = f1_score(
    oof_true, oof_preds,
    average="macro",
    zero_division=0,
)

print("=" * 60)
print(" OOF Evaluation — Qwen3-8B QLoRA (5-fold)")
print("=" * 60)
print(f"  Macro-F1 (classes 1–8) : {macro_f1_1_8:.4f}")
print(f"  Macro-F1 (classes 0–8) : {macro_f1_0_8:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# 8.2  Per-class report
# ---------------------------------------------------------------------------
print("\n--- Per-class F1 / Precision / Recall ---\n")
report = classification_report(
    oof_true,
    oof_preds,
    target_names=cfg.LABEL_NAMES,
    zero_division=0,
    digits=4,
)
print(report)

In [ ]:
# ---------------------------------------------------------------------------
# 8.3  Save evaluation results
# ---------------------------------------------------------------------------
results = {
    "model": cfg.MODEL_NAME_MAIN,
    "seed":  cfg.SEED,
    "n_folds": cfg.N_FOLDS,
    "oof_macro_f1_1_8": float(macro_f1_1_8),
    "oof_macro_f1_0_8": float(macro_f1_0_8),
}
results_path = RESULTS_DIR / "stage1_qwen3_8b_oof_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Results saved → {results_path}")
print(json.dumps(results, indent=2))

---
## Summary of Saved Artefacts

| File | Shape | Description |
|------|-------|-------------|
| `stage1_qwen3_8b/seed_42/oof_logits.npy` | `[1864, 9]` | OOF raw logits (unbiased) |
| `stage1_qwen3_8b/seed_42/oof_hidden.npy` | `[1864, D]` | OOF last-token hidden states |
| `stage1_qwen3_8b/seed_42/test_logits_by_fold.npy` | `[5, 472, 9]` | Per-fold test logits |
| `stage1_qwen3_8b/seed_42/test_logits_routed.npy` | `[472, 9]` | Held-out-routed test logits |
| `stage1_qwen3_8b/seed_42/test_logits_5fold_avg.npy` | `[472, 9]` | 5-fold average test logits (inflated baseline) |
| `stage1_qwen3_8b/seed_42/fold_k_best/` | — | LoRA adapter checkpoint for fold k |
| `results/stage1_qwen3_8b_oof_results.json` | — | Scalar evaluation metrics |
| `results/label_distribution.png` | — | *(from 01_eda_baseline.ipynb)* |

These artefacts are consumed by **`03_stage2_sequential.ipynb`** (stage-2 sequential classifier).